# 🎙️ Sleep2K - Phase 1: Benchmark & Test Workflow Voice Cloning

Notebook này được thiết kế độc lập 100% để chạy trên Google Colab (GPU Tesla T4 16GB VRAM Miễn Phí).
**Mục tiêu:** Đo đạc số liệu kỹ thuật thực tế (thời gian generate, VRAM, RTF) và chất lượng nghe thử âm thanh trước khi đưa ra bất kỳ quyết định tích hợp nào.

--- 
### Quy Trình Thử Nghiệm Gồm 2 Vòng:
- **Vòng A — Benchmark Model Thuần Túy:** `Audio mẫu + Text mẫu thủ công -> F5-TTS -> Đo thông số & Nghe thử`.
- **Vòng B — Test Đúng Workflow Sleep2K:** `Audio mẫu -> Whisper ASR tự nhận diện transcript -> F5-TTS -> Text mới -> Output`. Đánh giá xem ASR tự động có ảnh hưởng đến chất lượng clone không.

## Bước 1: Khởi tạo phần cứng & Đo VRAM ban đầu

In [ ]:
!nvidia-smi
import torch
print(f"CUDA khả dụng: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Thiết bị: {torch.cuda.get_device_name(0)}")
    print(f"Bộ nhớ GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Bước 2: Cài đặt Dependencies (F5-TTS & Whisper ASR)

In [ ]:
!pip install -q --upgrade pip
!pip install -q f5-tts openai-whisper soundfile torchaudio librosa tabulate
print("✅ Đã cài đặt xong thư viện!")

## Bước 3: Hàm tiện ích theo dõi VRAM & Đo đạc hiệu năng

In [ ]:
import time
import torch
import soundfile as sf
import os
from tabulate import tabulate
import IPython.display as ipd

def get_gpu_vram_mb():
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / (1024 * 1024)
    return 0

def reset_vram_peak():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

# Bảng ghi nhận kết quả đo đạc
benchmark_results = []

print("✅ Đã tải xong helper đo VRAM!")

## Bước 4: Tải âm thanh mẫu (Reference Audio)
Tải lên 1 hoặc nhiều file audio mẫu (.wav hoặc .mp3) có độ dài khoảng 10s, 20s hoặc 30s.

In [ ]:
from google.colab import files
print("👉 Bấm nút 'Choose Files' dưới đây để upload file audio mẫu:")
uploaded = files.upload()

ref_audio_list = list(uploaded.keys())
if ref_audio_list:
    primary_ref = ref_audio_list[0]
    info = sf.info(primary_ref)
    print(f"\n📁 Đã tải: {primary_ref} | Độ dài: {info.duration:.2f}s | Sample Rate: {info.samplerate}Hz")
    ipd.display(ipd.Audio(primary_ref))
else:
    print("Chưa chọn file nào.")

## ==========================================================
## VÒNG A: BENCHMARK MODEL THUẦN TÚY (MANUAL REFERENCE TEXT)
## ==========================================================
Đo thời gian generate, peak VRAM, và chất lượng âm thanh khi transcript mẫu được cung cấp chuẩn xác 100%.

In [ ]:
# --- CẤU HÌNH THỬ NGHIỆM VÒNG A ---
# 1. Điền text mà người nói đã phát âm trong audio mẫu:
manual_ref_text = "Nhập chính xác câu nói trong audio mẫu của bạn ở đây"

# 2. Danh sách các bài test theo ma trận (Tiếng Việt & Tiếng Trung):
test_cases_A = [
    {
        "id": "VI-01",
        "lang": "Tiếng Việt (Ngắn ~10s)",
        "text": "Chào bạn, đây là thử nghiệm nhân bản giọng nói AI đầu tiên trên hệ thống Sleep2K."
    },
    {
        "id": "VI-02",
        "lang": "Tiếng Việt (Dài ~30s)",
        "text": "Mục tiêu của chúng ta là xây dựng một nền tảng chuyển văn bản thành giọng nói và nhận dạng âm thanh hoàn chỉnh. Hệ thống này đảm bảo sự tách biệt độc lập giữa máy chủ điều khiển và các đơn vị tính toán AI GPU để tránh quá tải và đạt hiệu năng cao nhất."
    },
    {
        "id": "ZH-01",
        "lang": "Tiếng Trung (Ngắn)",
        "text": "你好，这是人工智能语音克隆系统的测试。"
    }
]

os.makedirs("output_vong_a", exist_ok=True)

for tc in test_cases_A:
    test_id = tc["id"]
    gen_text = tc["text"]
    out_file = f"output_vong_a/{test_id}.wav"
    
    print(f"\n--------------------------------------------------")
    print(f"▶️ Đang chạy test {test_id} [{tc['lang']}]...")
    print(f"Nội dung sinh: {gen_text}")
    
    reset_vram_peak()
    t_start = time.time()
    
    # Chạy F5-TTS CLI
    cmd = f'f5-tts_infer-cli --model "F5-TTS" --ref_audio "{primary_ref}" --ref_text "{manual_ref_text}" --gen_text "{gen_text}" --output_file "{out_file}"'
    exit_code = os.system(cmd)
    
    t_elapsed = time.time() - t_start
    peak_vram = get_gpu_vram_mb()
    
    out_dur = 0
    if os.path.exists(out_file):
        out_dur = sf.info(out_file).duration
        rtf = t_elapsed / out_dur if out_dur > 0 else 0
        print(f"✅ {test_id} xong: Thời gian gen: {t_elapsed:.2f}s | Audio ra: {out_dur:.2f}s | RTF: {rtf:.2f} | Peak VRAM: {peak_vram:.1f} MB")
        ipd.display(ipd.Audio(out_file))
        benchmark_results.append({
            "Vòng": "Vòng A (Manual Text)",
            "Mã Test": test_id,
            "Ngôn Ngữ": tc["lang"],
            "Thời Gian Gen (s)": f"{t_elapsed:.2f}",
            "Độ Dài Audio (s)": f"{out_dur:.2f}",
            "RTF": f"{rtf:.2f}",
            "VRAM Đỉnh (MB)": f"{peak_vram:.1f}"
        })
    else:
        print(f"❌ Lỗi sinh âm thanh cho {test_id}")

## ==========================================================
## VÒNG B: TEST ĐÚNG WORKFLOW SLEEP2K (AUTO ASR -> VOICE CLONE)
## ==========================================================
Trong quy trình này, người dùng chỉ upload audio. Whisper ASR sẽ tự động bóc tách text mẫu, sau đó chuyển thẳng sang F5-TTS để nhân bản giọng mới.

In [ ]:
import whisper

print("⏳ Đang nạp mô hình Whisper ASR (base)...")
asr_model = whisper.load_model("base")

print(f"🎯 Đang nhận diện tự động lời thoại trong: {primary_ref}...")
t_asr_start = time.time()
asr_result = asr_model.transcribe(primary_ref)
t_asr_elapsed = time.time() - t_asr_start

auto_ref_text = asr_result["text"].strip()
detected_lang = asr_result.get("language", "unknown")

print(f"\n=== KẾT QUẢ ASR TỰ ĐỘNG ===")
print(f"Thời gian ASR: {t_asr_elapsed:.2f}s")
print(f"Ngôn ngữ nhận diện: {detected_lang}")
print(f"Nội dung bóc tách được: \"{auto_ref_text}\"")
print("===========================")

# Tiến hành chạy Clone với text vừa bóc tách tự động
out_file_b = "output_vong_b/VI_AUTO_01.wav"
os.makedirs("output_vong_b", exist_ok=True)
gen_text_b = "Chào bạn, đây là kiểm thử quy trình khép kín tự động nhận diện chữ từ giọng nói và nhân bản trên Sleep2K."

reset_vram_peak()
t_clone_start = time.time()

cmd_b = f'f5-tts_infer-cli --model "F5-TTS" --ref_audio "{primary_ref}" --ref_text "{auto_ref_text}" --gen_text "{gen_text_b}" --output_file "{out_file_b}"'
os.system(cmd_b)

t_clone_elapsed = time.time() - t_clone_start
peak_vram_b = get_gpu_vram_mb()

if os.path.exists(out_file_b):
    dur_b = sf.info(out_file_b).duration
    total_pipeline_time = t_asr_elapsed + t_clone_elapsed
    rtf_b = t_clone_elapsed / dur_b if dur_b > 0 else 0
    print(f"\n✅ HOÀN THÀNH WORKFLOW KHÉP KÍN:")
    print(f"- Thời gian ASR: {t_asr_elapsed:.2f}s")
    print(f"- Thời gian Clone: {t_clone_elapsed:.2f}s")
    print(f"- Tổng thời gian End-to-End: {total_pipeline_time:.2f}s")
    print(f"- Audio sinh ra: {dur_b:.2f}s | RTF: {rtf_b:.2f} | Peak VRAM: {peak_vram_b:.1f} MB")
    ipd.display(ipd.Audio(out_file_b))
    benchmark_results.append({
        "Vòng": "Vòng B (Auto ASR -> F5)",
        "Mã Test": "VI-AUTO-01",
        "Ngôn Ngữ": "Tiếng Việt (Auto ASR)",
        "Thời Gian Gen (s)": f"{t_clone_elapsed:.2f} (ASR: {t_asr_elapsed:.2f}s)",
        "Độ Dài Audio (s)": f"{dur_b:.2f}",
        "RTF": f"{rtf_b:.2f}",
        "VRAM Đỉnh (MB)": f"{peak_vram_b:.1f}"
    })
else:
    print("❌ Lỗi sinh âm thanh ở Vòng B")

## ==========================================================
## BƯỚC 5: TỔNG HỢP BẢNG SỐ LIỆU ĐO ĐẠC THỰC TẾ (BENCHMARK REPORT)
## ==========================================================
Chạy cell dưới đây để in ra bảng tổng kết số liệu thực tế đo được để đối chiếu và đánh giá.

In [ ]:
if benchmark_results:
    headers = list(benchmark_results[0].keys())
    rows = [[r[k] for k in headers] for r in benchmark_results]
    print(tabulate(rows, headers=headers, tablefmt="github"))
else:
    print("Chưa có dữ liệu benchmark nào được ghi nhận.")